# Mouna — download remaining MS-ASL videos (Colab)

Runs the full download + keypoint-extraction pipeline on Colab's network instead of your home connection — this may sidestep the YouTube bot-detection rate limit we hit locally (different IP), though datacenter IPs can occasionally trip it too, so this isn't guaranteed.

**What this does, end to end**: fetches the official MS-ASL dataset metadata directly from Microsoft, downloads all 4,149 target clips fresh (simpler than trying to sync which ~2,027 we already have locally), extracts raw MediaPipe keypoints, and packages up just the small processed parquet for you to download back — never transfers the actual video files, which would be many GB.

**Runtime**: CPU is fine (keypoint extraction isn't GPU-accelerated in our pipeline) — no need to burn GPU quota here. `Runtime -> Change runtime type -> CPU`.

**Time**: likely 1-3 hours depending on Colab's network and whether the rate limit triggers. Colab Pro sessions run up to 24h, so this fits in one sitting — just don't close the tab.

In [ ]:
# 1. Clone the repo (has our download/extraction scripts; dataset metadata is NOT in git — fetched next)
!git clone https://github.com/vimalthomas/mouna.git
%cd mouna

In [ ]:
# 2. Install deps: yt-dlp, Node.js (yt-dlp needs a JS runtime for YouTube's signature decryption),
# and the MediaPipe extraction stack. ffmpeg is already on Colab images.
!pip install -q yt-dlp mediapipe opencv-python-headless pandas pyarrow loguru
!apt-get -qq update && apt-get -qq install -y nodejs > /dev/null
!yt-dlp --version
!node --version
!ffmpeg -version | head -1

In [ ]:
# 3. Fetch the OFFICIAL MS-ASL dataset metadata directly from Microsoft — NOT in git (it's real
# dataset content, correctly excluded by .gitignore's data/raw/ rule), so every fresh clone needs
# this step. This is the exact same source file used to build our local dataset.
!mkdir -p data/raw/msasl_official
!curl -sL -o /tmp/MS-ASL.zip "https://download.microsoft.com/download/3/c/a/3ca92c78-1c4a-4a91-a7ee-6980c1d242ec/MS-ASL.zip"
!unzip -o -q /tmp/MS-ASL.zip -d /tmp/MS-ASL_extracted
!cp /tmp/MS-ASL_extracted/MS-ASL/MSASL_classes.json /tmp/MS-ASL_extracted/MS-ASL/MSASL_train.json \
    /tmp/MS-ASL_extracted/MS-ASL/MSASL_val.json /tmp/MS-ASL_extracted/MS-ASL/MSASL_test.json \
    data/raw/msasl_official/
!ls -la data/raw/msasl_official/

In [ ]:
# 4. Confirm the target vocabulary is present
import json
signs = json.load(open('app/backend/signs.json'))
print('target vocabulary:', len(signs['signs']), 'signs')

In [ ]:
# 5. Run the download. Uses the same batch-then-pause logic that got us to ~2,900 real requests
# locally before the block reappeared (vs ~1,400-2,400 without it) — gentler by default, tune if needed.
# This prints live output; if it stalls on repeated "Sign in to confirm you're not a bot" errors,
# interrupt the cell (Runtime > Interrupt execution) rather than letting it burn through the whole list failing.
import os
os.environ['MOUNA_MSASL_BATCH_SIZE'] = '300'
os.environ['MOUNA_MSASL_PAUSE_SECONDS'] = '120'

!python3 scripts/download_msasl_official.py 2>&1 | tee /content/download_log.txt

In [ ]:
# 6. Check real results before spending time on extraction
import json
manifest = json.load(open('data/raw/msasl_official/videos/manifest.json'))
print(f'Downloaded: {len(manifest)} videos')

!grep -c "Sign in to confirm" /content/download_log.txt || echo "0 bot-check errors — clean run"

In [ ]:
# 7. Extract RAW (pre-normalization) keypoints — matches our local pipeline, so the per-part
# normalization + motion/orientation features can be applied identically afterward on your Mac.
import sys
sys.path.insert(0, 'scripts')
import mouna.data.preprocessing as prep
prep.KeypointExtractor.normalize_keypoints = lambda self, x: x  # identity — save raw, not z-scored

import os
os.environ['MOUNA_EXTRACT_MANIFESTS'] = 'data/raw/msasl_official/videos/manifest.json'
os.environ['MOUNA_EXTRACT_OUTPUT'] = 'data/silver/keypoints/keypoints_msasl_raw_colab.parquet'

import extract_keypoints
extract_keypoints.main()

In [ ]:
# 8. Package up just the small outputs (parquet + manifest) and download — never the raw videos
!zip -r msasl_download_results.zip data/silver/keypoints/keypoints_msasl_raw_colab.parquet data/raw/msasl_official/videos/manifest.json
!ls -lh msasl_download_results.zip

from google.colab import files
files.download('msasl_download_results.zip')

## After downloading `msasl_download_results.zip` to your Mac

Unzip it, then on your Mac:
```bash
cd ~/mouna/mouna
unzip ~/Downloads/msasl_download_results.zip -d /tmp/msasl_colab
cp /tmp/msasl_colab/data/silver/keypoints/keypoints_msasl_raw_colab.parquet data/silver/keypoints/keypoints_msasl_raw.parquet
cp /tmp/msasl_colab/data/raw/msasl_official/videos/manifest.json data/raw/msasl_official/videos/manifest.json
```
Then re-run `scripts/build_official_split_v2norm.py` (or the v3motion build step) to fold the new data into a fresh training set, and retrain.